# March Madness 2026 — EDA

Exploratory data analysis of all competition datasets.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import data_loader as dl, config
sns.set_style('whitegrid')
%matplotlib inline

## 1. Dataset Shapes

In [ ]:
for gender in ('M', 'W'):
    reg = dl.load_regular_season(gender)
    det = dl.load_regular_season(gender, detailed=True)
    trn = dl.load_tourney_results(gender)
    print(f'{gender} regular compact : {reg.shape}, {reg.Season.min()}-{reg.Season.max()}')
    print(f'{gender} regular detailed: {det.shape}, {det.Season.min()}-{det.Season.max()}')
    print(f'{gender} tourney compact : {trn.shape}, {trn.Season.min()}-{trn.Season.max()}')
    print()
massey = dl.load_massey_ordinals()
print(f'Massey: {massey.shape}, {massey.SystemName.nunique()} systems')

## 2. Score Distributions

In [ ]:
reg_m = dl.load_regular_season('M')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(reg_m['WScore'], bins=50, alpha=0.7, label='Winner')
axes[0].hist(reg_m['LScore'], bins=50, alpha=0.7, label='Loser')
axes[0].set_title('M Score Distribution (regular season)')
axes[0].legend()
margin = reg_m['WScore'] - reg_m['LScore']
axes[1].hist(margin, bins=50)
axes[1].set_title(f'Margin of Victory (mean={margin.mean():.1f})')
plt.tight_layout()
plt.savefig('../artifacts/score_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Seed vs Tourney Win Rate

In [ ]:
seeds_m = dl.load_tourney_seeds('M')
tourney_m = dl.load_tourney_results('M')
seed_map = seeds_m.set_index(['Season', 'TeamID'])['SeedNum']
wins, games = {}, {}
for _, row in tourney_m[tourney_m.Season >= 2010].iterrows():
    ws = seed_map.get((row.Season, row.WTeamID))
    ls = seed_map.get((row.Season, row.LTeamID))
    if ws is not None: wins[ws] = wins.get(ws,0)+1; games[ws] = games.get(ws,0)+1
    if ls is not None: games[ls] = games.get(ls,0)+1

seeds_list = list(range(1,17))
win_pcts = [wins.get(s,0)/games.get(s,1)*100 for s in seeds_list]
plt.figure(figsize=(10,4))
plt.bar(seeds_list, win_pcts)
plt.xlabel('Seed'); plt.ylabel('Win %'); plt.title('Seed Win Rate in NCAA Tournament (M, 2010-2024)')
plt.xticks(seeds_list)
plt.savefig('../artifacts/seed_win_rates.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Massey System Coverage

In [ ]:
massey = dl.load_massey_ordinals()
top_systems = massey.groupby('SystemName')['Season'].nunique().sort_values(ascending=False).head(20)
print('Top 20 systems by season coverage:')
print(top_systems.to_string())

## 5. Key Findings

- **M regular season**: 196,823 games, 1985–2026
- **W regular season**: 140,825 games, 1998–2026  
- **Massey ordinals**: 5.76M rows, 196 systems, 2003–2026
- **Score range**: Winner avg 77.0, Loser avg 64.9, margin avg 12.1
- **Seed win rates**: 1-seed=79.5%, 16-seed=25.8% (inflated by First Four play-in 16v16 games)
- **Missing data**: M detailed stats from 2003+, W from 2010+, Massey M-only, Coaches M-only
- **All referential integrity checks pass** — see artifacts/data_quality_report.txt